In [1]:
import os
import sys
import pandas as pd
from pathlib import Path

# Ensure project root is on sys.path so 'backtester' imports work
cwd = Path(os.getcwd())
candidates = [cwd, cwd.parent, Path("..").resolve()]
for p in candidates:
    if (p / "backtester").exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        break

print("PYTHONPATH set. Using root:", sys.path[0])

PYTHONPATH set. Using root: c:\Users\User\Desktop\Crypto strategy backtest


In [2]:
def _make_df(bo_rows: int = 10, pd_rows = 3, second_leg_rows = 10, side: str = "long") -> pd.DataFrame:
    idx = pd.date_range("2026-01-01", periods=bo_rows+pd_rows+second_leg_rows, freq="5min")
    if side == "long":
        df = pd.DataFrame([])
        df_BO = pd.DataFrame(
            {
                "open":  [100.0 + i for i in range(bo_rows)],
                "high":  [101.0 + i for i in range(bo_rows)],
                "low":   [99.0 + i for i in range(bo_rows)],
                "close": [100.5 + i for i in range(bo_rows)],
            },
            index=idx[:bo_rows],
        )
        BO_last_close = df_BO["close"].iat[-1]
        df_PB = pd.DataFrame(
            {
                "open":  [BO_last_close - i for i in range(pd_rows)],
                "high":  [BO_last_close + 1 - i for i in range(pd_rows)],
                "low":   [BO_last_close - 1 - i for i in range(pd_rows)],
                "close": [BO_last_close - 0.5 - i for i in range(pd_rows)],
            },
            index=idx[bo_rows:bo_rows+pd_rows],
        )
        pb_last_close = df_PB["close"].iat[-1]
        df_second_leg = pd.DataFrame(
            {
                "open":  [pb_last_close + i for i in range(second_leg_rows)],
                "high":  [pb_last_close + 1 + i for i in range(second_leg_rows)],
                "low":   [pb_last_close - 1 + i for i in range(second_leg_rows)],
                "close": [pb_last_close + 0.5 + i for i in range(second_leg_rows)],
            },
            index=idx[bo_rows+pd_rows:bo_rows+pd_rows+second_leg_rows],
        )
        df = pd.concat([df_BO, df_PB, df_second_leg], ignore_index=True)
        return df
    else:
        df = pd.DataFrame([])
        df_BO = pd.DataFrame(
            {
                "open":  [100.0 - i for i in range(bo_rows)],
                "high":  [101.0 - i for i in range(bo_rows)],
                "low":   [99.0 - i for i in range(bo_rows)],
                "close": [99.5 - i for i in range(bo_rows)],
            },
            index=idx[:bo_rows],
        )
        BO_last_close = df_BO["close"].iat[-1]
        df_PB = pd.DataFrame(
            {
                "open":  [BO_last_close + i for i in range(pd_rows)],
                "high":  [BO_last_close + 1 + i for i in range(pd_rows)],
                "low":   [BO_last_close - 1 + i for i in range(pd_rows)],
                "close": [BO_last_close + 0.5 + i for i in range(pd_rows)],
            },
            index=idx[bo_rows:bo_rows+pd_rows],
        )
        pb_last_close = df_PB["close"].iat[-1]
        df_second_leg = pd.DataFrame(
            {
                "open":  [pb_last_close - i for i in range(second_leg_rows)],
                "high":  [pb_last_close + 1 - i for i in range(second_leg_rows)],
                "low":   [pb_last_close - 1 - i for i in range(second_leg_rows)],
                "close": [pb_last_close - 0.5 - i for i in range(second_leg_rows)],
            },
            index=idx[bo_rows+pd_rows:bo_rows+pd_rows+second_leg_rows],
        )
        df = pd.concat([df_BO, df_PB, df_second_leg], ignore_index=True)
        return df

In [3]:

def load_crypto_parquet_data(coin_name: str, timeframe: str = "5m", nM: int = 54, section: str = "UTC") -> pd.DataFrame:
    df = pd.read_parquet(fr'C:\Users\User\Desktop\Crypto\{coin_name}_{timeframe}_{nM}M_{section}.parquet')
    return df
def generate_us_session_bars_info(df, include_holidays: bool = False):
    # 確保時間有時區資訊
    df['dt_ny'] = pd.to_datetime(df['dt_utc'], utc=True).dt.tz_convert('America/New_York')

    # 取日期（當地日曆）
    df['date'] = df['dt_ny'].dt.date
    df['weekday'] = df['dt_ny'].dt.day_name() 
    # 對每天依時間排序並編號
    df = df.sort_values(['date', 'dt_ny']).reset_index(drop=True)
    df['bar_index'] = df.groupby('date').cumcount() + 1  # 第幾根K線，從1開始
    if not include_holidays:
        weekday = ['Monday','Tuesday','Wednesday','Thursday','Friday']
    else:
        weekday = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    df = df.loc[df['weekday'].isin(weekday)]
    # 查看結果
    df.set_index('dt_utc', inplace=True)
    # print(df[['dt_ny', 'date', 'weekday', 'bar_index']].head(5))

    return df
def generate_allday_bars_info(df, include_holidays: bool = True):
# 確保時間有時區資訊
    df['dt_ny'] = pd.to_datetime(df['dt_utc'], utc=True).dt.tz_convert('America/New_York')

    # 取日期（當地日曆）
    df['date'] = df['dt_ny'].dt.date
    df['time'] = df['dt_ny'].dt.time
    df['weekday'] = df['dt_ny'].dt.day_name() 
    # 對每天依時間排序並編號
    df = df.sort_values(['date', 'dt_ny']).reset_index(drop=True)
    df['bar_index'] = df.groupby('date').cumcount() + 1  # 第幾根K線，從1開始
    if not include_holidays:
        weekday = ['Monday','Tuesday','Wednesday','Thursday','Friday']
    else:
        weekday = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    df = df.loc[df['weekday'].isin(weekday)]
    # 查看結果
    df.set_index('dt_utc', inplace=True)
    # print(df[['dt_ny', 'date', 'weekday', 'bar_index']].head(5))

    return df


In [8]:
df = load_crypto_parquet_data('ETH', timeframe='5m', nM=48, section='UTC')
df = generate_allday_bars_info(df, include_holidays=True)
df = df.loc['2025-12-31':'2025-12-31']  # 只取部分日期範圍

In [9]:
import backtester.indicators as idc

# df = _make_df(bo_rows=10, pd_rows=3, second_leg_rows=10, side="long")
df["ll_streak"] = idc.hh_ll_streak(df, side="ll")
# df["hh_streak"] = idc.hh_ll_streak(df, side="hh")
# df["ll_check"] = idc.hh_ll_check(df, side="ll")
# df["hh_check"] = idc.hh_ll_check(df, side="hh")
# df["pivot_low_mask"] = idc.pivot_mask(df, "low", 3)
df["efficiency_ratio"] = idc.efficiency_ratio(df, length=40, ema_length=3, column="close")
df["liner_regression_mid"] = idc.liner_regression_mid(df, length=100, column="close")
df["liner_regression_residuals"] = idc.liner_regression_residuals(df, length=100, column="close")
df["liner_regression_residuals_std"] = idc.bar_regression_residuals_std(df, length=100, column="close")
df["upper_boundary_outer"] = df["liner_regression_mid"] + 2 * df["liner_regression_residuals_std"]
df["upper_boundary_inner"] = df["liner_regression_mid"] + 1 * df["liner_regression_residuals_std"]
df["lower_boundary_outer"] = df["liner_regression_mid"] - 2 * df["liner_regression_residuals_std"]
df["lower_boundary_inner"] = df["liner_regression_mid"] - 1 * df["liner_regression_residuals_std"]

# upper_boundary_outer = float(regression_mid_series.iat[i]) + self.p.std_n * regression_residual_std
# upper_boundary_inner = float(regression_mid_series.iat[i]) + (self.p.std_n/2) * regression_residual_std
# lower_boundary_outer = float(regression_mid_series.iat[i]) - self.p.std_n * regression_residual_std
# lower_boundary_inner = float(regression_mid_series.iat[i]) - (self.p.std_n/2) * regression_residual_std

df[["low", "ll_streak"]]

,low,ll_streak
dt_utc,,
2025-12-31 00:00:00+00:00,2966.56,0
2025-12-31 00:05:00+00:00,2964.08,1
2025-12-31 00:10:00+00:00,2964.00,2
2025-12-31 00:15:00+00:00,2961.16,3
2025-12-31 00:20:00+00:00,2962.48,0
...,...,...
2025-12-31 23:35:00+00:00,2970.07,2
2025-12-31 23:40:00+00:00,2970.98,0
2025-12-31 23:45:00+00:00,2970.72,1
